In [2]:
import graph_helper_functions
from graph_helper_functions import *
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import igraph as ig
from igraph import plot
from sympy.abc import x, y
from qldpc import codes, circuits
from qldpc.objects import Pauli
import numpy as np
from collections import defaultdict, deque

In [3]:
import sys, shutil, importlib

sys.path.insert(0, "/mnt/data")

import graph_helper_functions as gh
importlib.reload(gh)

from graph_helper_functions import *

In [4]:
# construct tanner graph from merged codes
import numpy as np
import json
from graph_helper_functions import deform_code_for_logical
from graph_helper_functions import split_heavy_cycles
import numpy as np

def tanner_graph_from_deformations(
    deformation_result1,
    deformation_result2,
    basis="X",
    coords=None,
):
    merge_result = auxiliary_merge(deformation_result1, deformation_result2)

    return tanner_graph_from_merge_result_remapped(
        merge_result=merge_result,
        deformation_result1=deformation_result1,
        deformation_result2=deformation_result2,
        basis=basis,
        coords=coords,
    )


def tanner_graph_from_merge_result_remapped(
    merge_result,
    deformation_result1,
    deformation_result2,
    basis="X",
    coords=None,
):
    n_data1 = deformation_result1["n_original_qubits"]
    n_data2 = deformation_result2["n_original_qubits"]
    n_edges1 = deformation_result1["n_edges"]
    n_edges2 = deformation_result2["n_edges"]

    n_q1 = deformation_result1["n_qubits"]
    n_q2 = deformation_result2["n_qubits"]

    n_adapter = merge_result["n_adapter_edges"]

    # Desired ordering:
    # [data1 | data2 | edges1 | edges2 | adapters]
    offset_data1 = 0
    offset_data2 = n_data1
    offset_edge1 = n_data1 + n_data2
    offset_edge2 = n_data1 + n_data2 + n_edges1
    offset_adapter = n_data1 + n_data2 + n_edges1 + n_edges2

    n_total = offset_adapter + n_adapter

    def classify_check_block(
        support,
        offset_data1,
        offset_edge1,
        offset_data2,
        offset_edge2,
        offset_adapter,
        n_data1,
        n_edges1,
        n_data2,
        n_edges2,
        n_adapter,
    ):
        support = set(map(int, support))

        ranges = {
            "code1_data": set(range(offset_data1, offset_data1 + n_data1)),
            "code2_data": set(range(offset_data2, offset_data2 + n_data2)),
            "code1_gadget": set(range(offset_edge1, offset_edge1 + n_edges1)),
            "code2_gadget": set(range(offset_edge2, offset_edge2 + n_edges2)),
            "adapter": set(range(offset_adapter, offset_adapter + n_adapter)),
        }

        touched = [name for name, r in ranges.items() if support & r]

        if touched == ["code1_data"]:
            return "code1"
        if touched == ["code2_data"]:
            return "code2"
        if "adapter" in touched:
            return "adapter_or_merge"
        if any("gadget" in t for t in touched):
            return "deformed_or_gadget"
        if "code1_data" in touched and "code2_data" in touched:
            return "code_merge"
        return "mixed"

    def map_column(j):
        # Original auxiliary_merge ordering:
        # [data1 | edges1 | data2 | edges2 | adapters]
        if j < n_data1:
            return offset_data1 + j
        elif j < n_q1:
            return offset_edge1 + (j - n_data1)
        elif j < n_q1 + n_data2:
            return offset_data2 + (j - n_q1)
        elif j < n_q1 + n_q2:
            return offset_edge2 + (j - n_q1 - n_data2)
        else:
            return offset_adapter + (j - n_q1 - n_q2)

    def matrix_to_supports(M):
        supports = []
        for row in np.asarray(M, dtype=np.uint8):
            cols = np.where(row == 1)[0]
            supports.append(np.array([map_column(int(j)) for j in cols], dtype=int))
        return supports

    H_basis_supports = matrix_to_supports(merge_result["H_basis"])
    H_opposite_supports = matrix_to_supports(merge_result["H_opposite_basis"])

    if basis == "X":
        X_supports = H_basis_supports
        Z_supports = H_opposite_supports
    elif basis == "Z":
        Z_supports = H_basis_supports
        X_supports = H_opposite_supports
    else:
        raise ValueError("basis must be 'X' or 'Z'.")

    nodes = []
    links = []

    def get_coord(q_global):
        if coords is None:
            return (0.0, 0.0, 0.0)
        if q_global in coords:
            return coords[q_global]
        key = f"q_{q_global}"
        if key in coords:
            return coords[key]
        return (0.0, 0.0, 0.0)

    def add_qubit_node(q_global, qubit_type, block, original_index, fixed):
        x, y, z = get_coord(q_global)
        nodes.append({
            "id": f"q_{int(q_global)}",
            "kind": "qubit",
            "qubit_type": qubit_type,
            "block": block,
            "global_index": int(q_global),
            "original_index": int(original_index),
            "x": float(x),
            "y": float(y),
            "z": float(z),
            "fixed": bool(fixed),
        })

    # Qubits in desired order
    for i in range(n_data1):
        add_qubit_node(offset_data1 + i, "data", "code1", i, True)

    for i in range(n_data2):
        add_qubit_node(offset_data2 + i, "data", "code2", i, True)

    for e in range(n_edges1):
        add_qubit_node(offset_edge1 + e, "aux_edge", "code1_gadget", e, False)

    for e in range(n_edges2):
        add_qubit_node(offset_edge2 + e, "aux_edge", "code2_gadget", e, False)

    for a in range(n_adapter):
        add_qubit_node(offset_adapter + a, "adapter", "adapter", a, False)

    def add_check_nodes_and_links(supports, check_type):
        for r, support in enumerate(supports):
            support = np.asarray(support, dtype=int)
            check_id = f"{check_type.lower()}_{r}"

            nodes.append({
                "id": check_id,
                "kind": "check",
                "check_type": check_type,
                "block": classify_check_block(
                    support=support,
                    offset_data1=offset_data1,
                    offset_edge1=offset_edge1,
                    offset_data2=offset_data2,
                    offset_edge2=offset_edge2,
                    offset_adapter=offset_adapter,
                    n_data1=n_data1,
                    n_edges1=n_edges1,
                    n_data2=n_data2,
                    n_edges2=n_edges2,
                    n_adapter=n_adapter,
                ),
                "row_index": int(r),
                "weight": int(len(support)),
                "fixed": False,
            })

            for q in support:
                links.append({
                    "source": check_id,
                    "target": f"q_{int(q)}",
                    "interaction": "check_support",
                    "pauli": check_type,
                    "check_row": int(r),
                })

    add_check_nodes_and_links(X_supports, "X")
    add_check_nodes_and_links(Z_supports, "Z")

    return {
        "nodes": nodes,
        "links": links,
        "metadata": {
            "n_total_qubits": int(n_total),
            "n_data1": int(n_data1),
            "n_data2": int(n_data2),
            "n_edges1": int(n_edges1),
            "n_edges2": int(n_edges2),
            "n_adapter_edges": int(n_adapter),
            "ordering": ["data1", "data2", "edges1", "edges2", "adapter"],
            "offsets": {
                "data1": int(offset_data1),
                "data2": int(offset_data2),
                "edge1": int(offset_edge1),
                "edge2": int(offset_edge2),
                "adapter": int(offset_adapter),
            },
        },
    }

In [5]:
surface_code = codes.SurfaceCode(4, 4)
a = x**3+y+y**2
b = y**3+x+x**2
orders = {x: 6, y: 6}
BB_code = codes.BBCode(orders=orders, poly_a=a, poly_b=b)

code = BB_code
logical = code.get_logical_ops(Pauli.X)[0]
d1 = deform_code_for_logical(code.matrix, basis=Pauli.X, logical=logical)
d2 = deform_code_for_logical(code.matrix, basis=Pauli.X, logical=logical)
res = tanner_graph_from_deformations(d1, d2)


In [6]:
from collections import Counter
import numpy as np

nodes = res["nodes"]
links = res["links"]

qubits = [n for n in nodes if n["kind"] == "qubit"]
checks = [n for n in nodes if n["kind"] == "check"]

print("=" * 60)
print("TANNER GRAPH SUMMARY")
print("=" * 60)

print(f"total nodes   : {len(nodes)}")
print(f"qubit nodes   : {len(qubits)}")
print(f"check nodes   : {len(checks)}")
print(f"links         : {len(links)}")

print("\nqubit types:")
print(Counter(q["qubit_type"] for q in qubits))

print("\ncheck types:")
print(Counter(c["check_type"] for c in checks))

print("\ncheck blocks:")
print(Counter(c["block"] for c in checks))

print("\nfirst 5 nodes:")
for n in nodes[:5]:
    print(n)

print("\nfirst 5 links:")
for l in links[:5]:
    print(l)

print("=" * 60)

TANNER GRAPH SUMMARY
total nodes   : 351
qubit nodes   : 176
check nodes   : 175
links         : 1038

qubit types:
Counter({'data': 144, 'aux_edge': 24, 'adapter': 8})

check types:
Counter({'X': 88, 'Z': 87})

check blocks:
Counter({'code1': 60, 'code2': 60, 'deformed_or_gadget': 32, 'adapter_or_merge': 23})

first 5 nodes:
{'id': 'q_0', 'kind': 'qubit', 'qubit_type': 'data', 'block': 'code1', 'global_index': 0, 'original_index': 0, 'x': 0.0, 'y': 0.0, 'z': 0.0, 'fixed': True}
{'id': 'q_1', 'kind': 'qubit', 'qubit_type': 'data', 'block': 'code1', 'global_index': 1, 'original_index': 1, 'x': 0.0, 'y': 0.0, 'z': 0.0, 'fixed': True}
{'id': 'q_2', 'kind': 'qubit', 'qubit_type': 'data', 'block': 'code1', 'global_index': 2, 'original_index': 2, 'x': 0.0, 'y': 0.0, 'z': 0.0, 'fixed': True}
{'id': 'q_3', 'kind': 'qubit', 'qubit_type': 'data', 'block': 'code1', 'global_index': 3, 'original_index': 3, 'x': 0.0, 'y': 0.0, 'z': 0.0, 'fixed': True}
{'id': 'q_4', 'kind': 'qubit', 'qubit_type': 'da

In [8]:
logical_ops_x = np.where(BB_code.get_logical_ops(Pauli.X)[0]==1)
logical_ops_z = BB_code.get_logical_ops(Pauli.Z)

print(logical_ops_x)

(array([20, 23, 26, 29, 40, 41, 55, 56]),)


# Make visualisation for triangular lattice

In [18]:
# Export all 24 BB logical surgery layouts to JSON for the 3D HTML viewer.
# Run this cell, then open surgery_gadget_3d.html in a browser.

import collections
import itertools
import json
import math
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from qldpc.objects import Pauli
from tri_layout_deformation import tri_layout, _path_edges


def _bb_torus_geometry(BB_code, ell=6, m=6, R_major=80.0, r_minor=30.0):
    """Central BB code drawn on a torus: data/check nodes plus Tanner edges."""
    H = np.asarray(BB_code.matrix, dtype=np.uint8) % 2
    rows, cols = H.shape
    n_data = cols // 2
    n_sites = ell * m
    n_x = rows // 2

    def torus_xyz(u, v):
        theta = 2 * math.pi * u / (2 * m)
        phi = 2 * math.pi * v / (2 * ell)
        x0 = (R_major + r_minor * math.cos(phi)) * math.cos(theta)
        y0 = (R_major + r_minor * math.cos(phi)) * math.sin(theta)
        z0 = r_minor * math.sin(phi)
        return float(x0), float(y0), float(z0)

    def site_to_ab(i):
        return i // m, i % m

    nodes = []
    links = []

    for q in range(n_data):
        reg = "A" if q < n_sites else "B"
        site = q if q < n_sites else q - n_sites
        a, b = site_to_ab(site)
        u = 2 * b + (0 if reg == "A" else 1)
        v = 2 * a
        x0, y0, z0 = torus_xyz(u, v)
        nodes.append({
            "id": f"bb:data:{q}",
            "kind": "bb_data",
            "register": reg,
            "index": int(q),
            "x": x0,
            "y": y0,
            "z": z0,
        })

    for c in range(rows):
        check_type = "X" if c < n_x else "Z"
        local = c if c < n_x else c - n_x
        a, b = site_to_ab(local)
        u = 2 * b + 0.5
        v = 2 * a + (0.65 if check_type == "X" else -0.65)
        x0, y0, z0 = torus_xyz(u, v)
        nodes.append({
            "id": f"bb:check:{c}",
            "kind": "bb_check",
            "check_type": check_type,
            "index": int(c),
            "x": x0,
            "y": y0,
            "z": z0,
        })

    for c in range(rows):
        support = np.where(H[c] == 1)[0]
        for q in support:
            links.append({
                "source": f"bb:check:{c}",
                "target": f"bb:data:{int(q)}",
                "kind": "bb_tanner",
                "check_type": "X" if c < n_x else "Z",
            })

    return nodes, links


def _floor_basis(index, floor_count=24):
    """Radial panel basis vectors for one floor on the surrounding ring."""
    theta = 2 * math.pi * index / floor_count
    radial = np.array([math.cos(theta), math.sin(theta), 0.0])
    tangent = np.array([-math.sin(theta), math.cos(theta), 0.0])
    vertical = np.array([0.0, 0.0, 1.0])
    return radial, tangent, vertical


def _floor_radius(index, ring_radius=480.0):
    return ring_radius


def _world_from_floor(local_x, local_y, floor_index, radial_scale=230.0, angular_scale=150.0):
    radial, tangent, vertical = _floor_basis(floor_index)
    floor_radius = _floor_radius(floor_index)
    # Horizontal floor in the BB torus mid-plane. A large ring gives each of the
    # 24 floors enough angular room, so the floor does not need to be skinny.
    point = (floor_radius + radial_scale * local_x) * radial + angular_scale * local_y * tangent
    return [float(point[0]), float(point[1]), float(point[2])]


def _layout_local_position(site, lattice_pos, center, max_span):
    p = np.asarray(lattice_pos[site], dtype=float)
    centered = p - center
    return centered[0] / max_span, centered[1] / max_span


def _make_floor_record(layout, basis, logical_index, floor_index, max_port_count, max_span_hint=None):
    logical = np.asarray(layout["deformation_result"]["logical"], dtype=np.uint8)
    basis_name = basis.name if hasattr(basis, "name") else str(basis)

    state = layout["state"]
    stabilizers = layout["stabilizers"]
    lattice_graph = state["lattice_graph"]
    lattice_pos = state["lattice_pos"]
    materialized_edges = list(stabilizers["materialized_edges"])
    materialized_graph = stabilizers["materialized_graph"]
    vertex_sites = stabilizers["original_vertex_to_site"]

    pts = np.array(list(lattice_pos.values()), dtype=float)
    center = (pts.min(axis=0) + pts.max(axis=0)) / 2
    span = float(max(*(pts.max(axis=0) - pts.min(axis=0)), 1.0))
    if max_span_hint is not None:
        span = max(span, float(max_span_hint))

    floor_id = f"{basis_name}{logical_index}"
    floor_nodes = []
    floor_links = []

    def site_node_id(site):
        return f"floor:{floor_id}:site:{site[0]}:{site[1]}"

    vertex_to_qubit = {
        int(v): int(q)
        for q, v in layout["deformation_result"]["qubit_to_vertex"].items()
    }
    site_to_port = {
        vertex_sites[int(v)]: int(q)
        for v, q in vertex_to_qubit.items()
        if int(v) in vertex_sites
    }

    # Geometric triangular-lattice vertices. These are anchors; edge qubits live
    # on lattice edges, and stabilizer measurement nodes are added below.
    for site in lattice_graph.nodes():
        lx, ly = _layout_local_position(site, lattice_pos, center, span)
        x0, y0, z0 = _world_from_floor(lx, ly, floor_index)
        floor_nodes.append({
            "id": site_node_id(site),
            "kind": "lattice_vertex",
            "floor": floor_id,
            "basis": basis_name,
            "logical_index": int(logical_index),
            "site": [int(site[0]), int(site[1])],
            "x": x0,
            "y": y0,
            "z": z0,
        })

    # Every triangular-lattice edge is an edge qubit at the edge midpoint.
    materialized_edge_set = {tuple(sorted(edge)) for edge in materialized_edges}
    lattice_edge_to_qubit = stabilizers.get("lattice_edge_to_qubit", {})
    physical_qubit_to_edge_node = {}
    for edge_index, (a, b) in enumerate(lattice_graph.edges()):
        edge = tuple(sorted((a, b)))
        pa = np.asarray(lattice_pos[a], dtype=float)
        pb = np.asarray(lattice_pos[b], dtype=float)
        mid = (pa + pb) / 2
        lx, ly = (mid - center) / span
        x0, y0, z0 = _world_from_floor(lx, ly, floor_index)
        active = edge in materialized_edge_set
        edge_qubit_id = f"floor:{floor_id}:edgeq:{edge_index}"
        physical_qubit = int(lattice_edge_to_qubit[edge]) if active and edge in lattice_edge_to_qubit else None
        floor_nodes.append({
            "id": edge_qubit_id,
            "kind": "edge_qubit" if active else "unused_edge_qubit",
            "floor": floor_id,
            "basis": basis_name,
            "logical_index": int(logical_index),
            "edge_index": int(edge_index),
            "active": bool(active),
            "physical_qubit": physical_qubit,
            "x": x0,
            "y": y0,
            "z": z0,
        })
        if physical_qubit is not None:
            physical_qubit_to_edge_node[physical_qubit] = edge_qubit_id
        for endpoint in [a, b]:
            floor_links.append({
                "source": site_node_id(endpoint),
                "target": edge_qubit_id,
                "kind": "active_edge_incidence" if active else "lattice_edge_incidence",
                "floor": floor_id,
            })

    # Actual vertex checks from the lattice-lifted basis stabilizers.
    port_sites = {}
    vertex_check_sites = sorted(stabilizers.get("site_to_vertex_check_row", {}), key=str)
    for site in vertex_check_sites:
        lx, ly = _layout_local_position(site, lattice_pos, center, span)
        x0, y0, z0 = _world_from_floor(lx, ly, floor_index)
        qubit = site_to_port.get(site)
        check_id = f"floor:{floor_id}:vertex_check:{site[0]}:{site[1]}"
        floor_nodes.append({
            "id": check_id,
            "kind": "port_vertex_check" if qubit is not None else "vertex_check",
            "floor": floor_id,
            "basis": basis_name,
            "logical_index": int(logical_index),
            "bb_qubit": None if qubit is None else int(qubit),
            "site": [int(site[0]), int(site[1])],
            "x": x0,
            "y": y0,
            "z": z0,
        })
        if qubit is not None:
            port_sites[int(qubit)] = check_id
        for edge in lattice_graph.edges(site):
            e = tuple(sorted(edge))
            q = lattice_edge_to_qubit.get(e)
            edge_node = physical_qubit_to_edge_node.get(q)
            if edge_node is not None:
                floor_links.append({
                    "source": check_id,
                    "target": edge_node,
                    "kind": "vertex_check_incidence",
                    "floor": floor_id,
                })

    # Actual cycle checks from the opposite-basis new stabilizers.
    n_data = int(stabilizers["n_data_qubits"])
    H_cycle = np.asarray(stabilizers["H_opposite_basis_new"], dtype=np.uint8)
    for row_index, row in enumerate(H_cycle):
        support = [int(q) for q in np.where(row[n_data:] == 1)[0] + n_data]
        edge_nodes = [physical_qubit_to_edge_node[q] for q in support if q in physical_qubit_to_edge_node]
        if not edge_nodes:
            continue
        coords = np.array([
            [node["x"], node["y"], node["z"]]
            for node in floor_nodes
            if node["id"] in set(edge_nodes)
        ], dtype=float)
        centroid = coords.mean(axis=0)
        check_id = f"floor:{floor_id}:cycle_check:{row_index}"
        floor_nodes.append({
            "id": check_id,
            "kind": "cycle_check",
            "floor": floor_id,
            "basis": basis_name,
            "logical_index": int(logical_index),
            "cycle_row": int(row_index),
            "weight": int(len(edge_nodes)),
            "x": float(centroid[0]),
            "y": float(centroid[1]),
            "z": 0.0,
        })
        for edge_node in edge_nodes:
            floor_links.append({
                "source": check_id,
                "target": edge_node,
                "kind": "cycle_check_incidence",
                "floor": floor_id,
            })

    # Equal-sized boundary of port placeholders for all floors.
    radial, tangent, vertical = _floor_basis(floor_index)
    active_ports = sorted(port_sites.items())
    support_qubits = [q for q, _ in active_ports]
    boundary_radius = _floor_radius(floor_index) + 190.0
    for i in range(max_port_count):
        t = 0.0 if max_port_count == 1 else (i / (max_port_count - 1) - 0.5)
        point = boundary_radius * radial + 160.0 * t * tangent
        active = i < len(active_ports)
        q = support_qubits[i] if i < len(support_qubits) else None
        boundary_id = f"floor:{floor_id}:boundary_port:{i}"
        floor_nodes.append({
            "id": boundary_id,
            "kind": "boundary_port" if active else "unused_boundary_port",
            "floor": floor_id,
            "basis": basis_name,
            "logical_index": int(logical_index),
            "port_index": int(i),
            "bb_qubit": q,
            "active": bool(active),
            "x": float(point[0]),
            "y": float(point[1]),
            "z": float(point[2]),
        })
        if active:
            _, port_node_id = active_ports[i]
            floor_links.append({
                "source": port_node_id,
                "target": boundary_id,
                "kind": "port_to_boundary",
                "floor": floor_id,
            })
            if q is not None:
                floor_links.append({
                    "source": f"bb:data:{q}",
                    "target": boundary_id,
                    "kind": "bb_to_boundary_support",
                    "floor": floor_id,
                })

    return {
        "id": floor_id,
        "basis": basis_name,
        "logical_index": int(logical_index),
        "support_weight": int(np.sum(logical)),
        "rows": int(state["rows"]),
        "cols": int(state["cols"]),
        "nodes": floor_nodes,
        "links": floor_links,
    }


def _add_triangle_center_visuals(graph_data):
    """Convert floor display to edge qubits plus triangle-center measurements."""
    nodes = graph_data["nodes"]
    links = graph_data["links"]
    nodes_by_id = {node["id"]: node for node in nodes}

    def link_id(endpoint):
        return endpoint["id"] if isinstance(endpoint, dict) else endpoint

    for node in nodes:
        if node.get("kind") == "measurement_qubit" and ":site:" in node["id"]:
            node["kind"] = "lattice_vertex"

    edge_endpoints = collections.defaultdict(list)
    for link in links:
        if link.get("kind") not in {"lattice_edge_incidence", "active_edge_incidence"}:
            continue
        source = link_id(link["source"])
        target = link_id(link["target"])
        if source not in nodes_by_id or target not in nodes_by_id:
            continue
        if nodes_by_id[source]["kind"] in {"edge_qubit", "unused_edge_qubit"}:
            edge_qubit, site = source, target
        elif nodes_by_id[target]["kind"] in {"edge_qubit", "unused_edge_qubit"}:
            edge_qubit, site = target, source
        else:
            continue
        edge_endpoints[edge_qubit].append(site)

    floor_edges = collections.defaultdict(dict)
    floor_adj = collections.defaultdict(lambda: collections.defaultdict(set))
    for edge_qubit, sites in edge_endpoints.items():
        if len(sites) != 2:
            continue
        a, b = sorted(sites)
        floor = nodes_by_id[edge_qubit].get("floor")
        floor_edges[floor][(a, b)] = edge_qubit
        floor_adj[floor][a].add(b)
        floor_adj[floor][b].add(a)

    new_nodes = []
    new_links = []
    for floor, adj in floor_adj.items():
        triangles = set()
        for a in sorted(adj):
            for b, c in itertools.combinations(sorted(adj[a]), 2):
                if b in adj[c]:
                    triangles.add(tuple(sorted((a, b, c))))
        for i, tri in enumerate(sorted(triangles)):
            pts = np.array([[nodes_by_id[s]["x"], nodes_by_id[s]["y"], nodes_by_id[s]["z"]] for s in tri], dtype=float)
            lens = sorted(float(np.linalg.norm(pts[p] - pts[q])) for p, q in [(0, 1), (0, 2), (1, 2)])
            if lens[2] > 1.35 * lens[0]:
                continue
            centroid = pts.mean(axis=0)
            measurement_id = f"floor:{floor}:triangle_meas:{i}"
            new_nodes.append({
                "id": measurement_id,
                "kind": "triangle_measurement_qubit",
                "floor": floor,
                "basis": nodes_by_id[tri[0]].get("basis"),
                "logical_index": nodes_by_id[tri[0]].get("logical_index"),
                "triangle_index": int(i),
                "x": float(centroid[0]),
                "y": float(centroid[1]),
                "z": 0.0,
            })
            for a, b in itertools.combinations(sorted(tri), 2):
                edge_qubit = floor_edges[floor].get(tuple(sorted((a, b))))
                if edge_qubit:
                    new_links.append({
                        "source": measurement_id,
                        "target": edge_qubit,
                        "kind": "triangle_measurement_incidence",
                        "floor": floor,
                    })

    for edge_qubit, sites in edge_endpoints.items():
        if len(sites) == 2 and nodes_by_id[edge_qubit]["kind"] == "edge_qubit":
            new_links.append({
                "source": sites[0],
                "target": sites[1],
                "kind": "used_edge_highlight",
                "floor": nodes_by_id[edge_qubit].get("floor"),
                "edge_qubit": edge_qubit,
            })

    nodes.extend(new_nodes)
    links.extend(new_links)
    graph_data["metadata"]["floor_visual_model"] = "edge_qubits_triangle_measurements_v1"
    graph_data["metadata"]["num_nodes"] = len(nodes)
    graph_data["metadata"]["num_links"] = len(links)
    return graph_data

def export_surgery_gadget_atlas(BB_code, output_path="surgery_logical_gadgets.json"):
    """Write the 24-floor logical surgery gadget atlas used by the HTML viewer."""
    output_path = Path(output_path)
    z_logicals = BB_code.get_logical_ops(Pauli.Z)
    x_logicals = BB_code.get_logical_ops(Pauli.X)
    z_weights = [int(np.sum(np.asarray(logical, dtype=np.uint8))) for logical in z_logicals]
    x_weights = [int(np.sum(np.asarray(logical, dtype=np.uint8))) for logical in x_logicals]
    max_port_count = max(z_weights + x_weights)

    # Alternating floors: Z0, X0, Z1, X1, ...
    floor_specs = []
    for i in range(len(z_logicals)):
        floor_specs.append((Pauli.Z, i))
        floor_specs.append((Pauli.X, i))

    # First pass: build all layouts and find the largest lattice scale.
    # Every floor is then drawn with that same scale, so smaller logical gadgets
    # occupy a subset of the common floor size.
    layout_records = []
    max_span = 1.0
    max_rows = 0
    max_cols = 0
    for floor_index, (basis, logical_index) in enumerate(floor_specs):
        basis_name = basis.name if hasattr(basis, "name") else str(basis)
        print(f"building layout {floor_index + 1}/{len(floor_specs)}: {basis_name}{logical_index}")
        logical = BB_code.get_logical_ops(basis)[logical_index]
        def_res = deform_code_for_logical(BB_code.matrix, basis=basis, logical=logical)
        layout = tri_layout(def_res)
        plt.close(layout["fig"])
        pts = np.array(list(layout["state"]["lattice_pos"].values()), dtype=float)
        span = float(max(*(pts.max(axis=0) - pts.min(axis=0)), 1.0))
        max_span = max(max_span, span)
        max_rows = max(max_rows, int(layout["state"]["rows"]))
        max_cols = max(max_cols, int(layout["state"]["cols"]))
        layout_records.append((floor_index, basis, logical_index, layout))

    bb_nodes, bb_links = _bb_torus_geometry(BB_code)
    floors = []
    nodes = list(bb_nodes)
    links = list(bb_links)

    for floor_index, basis, logical_index, layout in layout_records:
        floor = _make_floor_record(
            layout,
            basis=basis,
            logical_index=logical_index,
            floor_index=floor_index,
            max_port_count=max_port_count,
            max_span_hint=max_span,
        )
        floor["max_rows"] = max_rows
        floor["max_cols"] = max_cols
        floors.append({k: v for k, v in floor.items() if k not in {"nodes", "links"}})
        nodes.extend(floor["nodes"])
        links.extend(floor["links"])

    graph_data = {
        "metadata": {
            "description": "BB code with 24 surrounding triangularized logical surgery gadgets",
            "num_floors": len(floors),
            "max_port_count": max_port_count,
            "max_rows": max_rows,
            "max_cols": max_cols,
            "num_nodes": len(nodes),
            "num_links": len(links),
        },
        "floors": floors,
        "nodes": nodes,
        "links": links,
    }
    graph_data = _add_triangle_center_visuals(graph_data)
    output_path.write_text(json.dumps(graph_data, indent=2))
    print(f"wrote {output_path.resolve()}")
    print(f"nodes={len(nodes)}, links={len(links)}, floors={len(floors)}")
    return graph_data


surgery_graph_data = export_surgery_gadget_atlas(
    BB_code,
    output_path="surgery_logical_gadgets.json",
)


building layout 1/24: Z0
building layout 2/24: X0
building layout 3/24: Z1
building layout 4/24: X1
building layout 5/24: Z2
building layout 6/24: X2
building layout 7/24: Z3
building layout 8/24: X3
building layout 9/24: Z4
building layout 10/24: X4
building layout 11/24: Z5
building layout 12/24: X5
building layout 13/24: Z6
building layout 14/24: X6
building layout 15/24: Z7
building layout 16/24: X7
building layout 17/24: Z8
building layout 18/24: X8
building layout 19/24: Z9
building layout 20/24: X9
building layout 21/24: Z10
building layout 22/24: X10
building layout 23/24: Z11
building layout 24/24: X11
wrote /Users/jeanettetorronen/Desktop/quantum_compiler/Ravioli-project/surgery_logical_gadgets.json
nodes=5204, links=11814, floors=24


In [24]:
from IPython.display import IFrame, display
from pathlib import Path
import http.server
import socket
import socketserver
import threading

viewer_dir = Path("/Users/jeanettetorronen/Desktop/quantum_compiler/Ravioli-project")
viewer_file = viewer_dir / "surgery_gadget_3d.html"
json_file = viewer_dir / "surgery_logical_gadgets.json"

if not viewer_file.exists():
    raise FileNotFoundError(viewer_file)
if not json_file.exists():
    raise FileNotFoundError("Run the exporter cell first: surgery_logical_gadgets.json was not found")

class _ViewerHandler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=str(viewer_dir), **kwargs)

    def log_message(self, format, *args):
        pass


def _find_free_port(start=8765, stop=8865):
    for port in range(start, stop):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) != 0:
                return port
    raise RuntimeError("Could not find a free localhost port")

if "surgery_viewer_server" not in globals():
    port = _find_free_port()
    surgery_viewer_server = socketserver.ThreadingTCPServer(("127.0.0.1", port), _ViewerHandler)
    surgery_viewer_server.daemon_threads = True
    surgery_viewer_thread = threading.Thread(target=surgery_viewer_server.serve_forever, daemon=True)
    surgery_viewer_thread.start()
    surgery_viewer_url = f"http://127.0.0.1:{port}/surgery_gadget_3d.html"
else:
    port = surgery_viewer_server.server_address[1]
    surgery_viewer_url = f"http://127.0.0.1:{port}/surgery_gadget_3d.html"

print("viewer:", surgery_viewer_url)
display(IFrame(surgery_viewer_url, width="100%", height=760))


viewer: http://127.0.0.1:8765/surgery_gadget_3d.html
